In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("..\data\champions_by_lane.csv")

MIN_PARTIDAS = 100
df = df[df["games_played"] >= MIN_PARTIDAS].reset_index(drop=True)

feature_cols = [
    "avg_kills", "avg_deaths", "avg_assists", "avg_kda",
    "avg_cs_at_10", "avg_gold_at_10", "avg_xp_at_10", "avg_damage_at_10",
    "avg_cs_at_15", "avg_gold_at_15", "avg_xp_at_15", "avg_damage_at_15",
    "avg_cs_at_20", "avg_gold_at_20", "avg_xp_at_20", "avg_damage_at_20",
    "avg_total_damage", "avg_damage_taken", "avg_gold_earned", "avg_total_cs",
    "avg_vision_score", "first_blood_rate_pct", "first_tower_rate_pct"
]

scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols])

champLaneVectors = {}
for i, row in df.iterrows():
    key = (row["champion_name"], row["lane"])
    champLaneVectors[key] = X[i]

laneVectors = {}
for lane in df["lane"].unique():
    mask = (df["lane"] == lane).values
    laneVectors[lane] = X[mask].mean(axis=0)



In [2]:
main_lane = df.loc[df.groupby("champion_name")["games_played"].idxmax()].set_index("champion_name")["lane"]

records = []
for (champ, lane), vec in champLaneVectors.items():
    if main_lane[champ] != lane:
        continue  # solo campeones cuya línea principal es esta (sin pockets off-role)
    dist = float(np.linalg.norm(vec - laneVectors[lane]))
    records.append({"champion": champ, "lane": lane, "dist_to_lane": dist})

dist_df = pd.DataFrame(records)
mean_dist_per_lane = dist_df.groupby("lane")["dist_to_lane"].mean().round(4)
print("Distancia media al vector de línea (mains, espacio estandarizado):")
print(mean_dist_per_lane.sort_values().to_string())

LANE_FOCO = "BOTTOM"  # ADC
foco = dist_df[dist_df["lane"] == LANE_FOCO].sort_values("dist_to_lane")
print(f"\n{LANE_FOCO}: {len(foco)} mains, media = {foco['dist_to_lane'].mean():.4f}")
print("Más cercanos al vector:", ", ".join(foco.head(5)["champion"]))
print("Más lejanos al vector:", ", ".join(foco.tail(5)["champion"]))
foco.reset_index(drop=True)

Distancia media al vector de línea (mains, espacio estandarizado):
lane
MIDDLE     2.1982
BOTTOM     2.3027
JUNGLE     2.3255
TOP        2.4004
UTILITY    3.1215

BOTTOM: 26 mains, media = 2.3027
Más cercanos al vector: Ashe, Jinx, Vayne, Yunara, MissFortune
Más lejanos al vector: Twitch, Samira, Ziggs, Kalista, Draven


,champion,lane,dist_to_lane
0,Ashe,BOTTOM,1.200159
1,Jinx,BOTTOM,1.405851
2,Vayne,BOTTOM,1.653748
3,Yunara,BOTTOM,1.704974
4,MissFortune,BOTTOM,1.733610
5,Xayah,BOTTOM,1.781226
6,Jhin,BOTTOM,1.956649
7,Varus,BOTTOM,1.966434
8,Aphelios,BOTTOM,2.021735
9,Smolder,BOTTOM,2.038925


In [3]:
ROLE_MIN_SHARE = 0.15  # rol primario: >=15% de sus partidas (presencia)

raw = pd.read_csv("..\data\champions_by_lane.csv")
tot = raw.groupby("champion_name")["games_played"].transform("sum")
raw["lane_share"] = raw["games_played"] / tot

# Roles primarios por campeón (presencia en crudo) con embedding disponible (>=100 part)
roles = {}
for champ, g in raw.groupby("champion_name"):
    prim = [r["lane"] for _, r in g.iterrows()
            if r["lane_share"] >= ROLE_MIN_SHARE and (champ, r["lane"]) in champLaneVectors]
    if len(prim) >= 2:
        roles[champ] = sorted(prim)

print(f"Flex picks (>=2 roles al {ROLE_MIN_SHARE:.0%}): {len(roles)}")

ana_records, base_records = [], []
for champ, lanes in roles.items():
    for A in lanes:
        for B in lanes:
            if A == B:
                continue
            pred = champLaneVectors[(champ, A)] - laneVectors[A] + laneVectors[B]
            err = float(np.linalg.norm(pred - champLaneVectors[(champ, B)]))
            ana_records.append({"champion": champ, "from": A, "to": B, "analogy_err": err})
            base_records.append(float(np.linalg.norm(
                champLaneVectors[(champ, A)] - champLaneVectors[(champ, B)])))

ana_df = pd.DataFrame(ana_records)
per_champ = ana_df.groupby("champion")["analogy_err"].mean().round(4).sort_values()
print(f"\nError medio de analogía (N = n° roles primarios por campeón): {ana_df['analogy_err'].mean():.4f}")
print(f"Baseline sin trasladar (emb_A vs emb_B directo): {np.mean(base_records):.4f}")
print("\nMejores analogías (traslación casi exacta):")
print(per_champ.head(10).to_string())
print("\nPeores analogías (el campeón cambia mucho según la línea):")
print(per_champ.tail(10).to_string())
per_champ

Flex picks (>=2 roles al 15%): 61

Error medio de analogía (N = n° roles primarios por campeón): 1.6339
Baseline sin trasladar (emb_A vs emb_B directo): 5.1655

Mejores analogías (traslación casi exacta):
champion
Riven         0.5166
Viktor        0.6471
Cassiopeia    0.6580
Anivia        0.7160
Akali         0.7795
Aurora        0.8230
Irelia        0.8262
Hwei          0.8539
Vayne         0.8796
Yone          0.8819

Peores analogías (el campeón cambia mucho según la línea):
champion
Jayce       2.3221
Pantheon    2.3485
Qiyana      2.4043
Karthus     2.4255
Brand       2.4300
Talon       2.7260
Trundle     2.7883
Senna       2.9797
Taliyah     3.1863
Camille     3.2811


champion
Riven         0.5166
Viktor        0.6471
Cassiopeia    0.6580
Anivia        0.7160
Akali         0.7795
               ...  
Talon         2.7260
Trundle       2.7883
Senna         2.9797
Taliyah       3.1863
Camille       3.2811
Name: analogy_err, Length: 61, dtype: float64

In [4]:
from sklearn.metrics import pairwise_distances

keys = list(champLaneVectors.keys())
M = np.stack([champLaneVectors[k] for k in keys])
D = pairwise_distances(M)  # matriz simétrica de distancias campeón-línea

iu = np.triu_indices(len(keys), k=1)
all_d = D[iu]
i_max = np.argmax(D)
a, b = keys[i_max // len(keys)], keys[i_max % len(keys)]
print(f"Pares de campeones: {len(all_d):,} | media = {all_d.mean():.4f} | mediana = {np.median(all_d):.4f}")
print(f"Máxima distancia: {D.max():.4f} entre {a[0]}({a[1]}) y {b[0]}({b[1]})")
print(f"Mínima distancia (no idénticos): {all_d[all_d > 1e-9].min():.4f}")

lanes = sorted(laneVectors.keys())
LV = np.stack([laneVectors[l] for l in lanes])
DL = pd.DataFrame(pairwise_distances(LV), index=lanes, columns=lanes).round(4)
print("\nDistancias entre vectores de línea:")
print(DL.to_string())

ana_mean = float(ana_df["analogy_err"].mean())
base_mean = float(np.mean(base_records))
azar_ref = float(np.sqrt(2 * M.shape[1]))
print(f"\nLectura: analogía {ana_mean:.4f} vs sin trasladar {base_mean:.4f} "
      f"vs media entre campeones {all_d.mean():.2f} (azar √(2d) ≈ {azar_ref:.2f}) "
      f"-> error aprox {100 * ana_mean / all_d.mean():.0f}% de una distancia típica")
print(f"Distancia media de mains a su línea (celda 2): {dict(mean_dist_per_lane.round(4))}")

Pares de campeones: 143,380 | media = 6.1828 | mediana = 5.4785
Máxima distancia: 18.1784 entre Milio(UTILITY) y Elise(TOP)
Mínima distancia (no idénticos): 0.5161

Distancias entre vectores de línea:
         BOTTOM  JUNGLE  MIDDLE     TOP  UTILITY
BOTTOM   0.0000  3.2644  2.6694  3.8502   8.2130
JUNGLE   3.2644  0.0000  4.0147  5.0997   7.6521
MIDDLE   2.6694  4.0147  0.0000  1.8331   9.1146
TOP      3.8502  5.0997  1.8331  0.0000   9.1036
UTILITY  8.2130  7.6521  9.1146  9.1036   0.0000

Lectura: analogía 1.6339 vs sin trasladar 5.1655 vs media entre campeones 6.18 (azar √(2d) ≈ 6.78) -> error aprox 26% de una distancia típica
Distancia media de mains a su línea (celda 2): {'BOTTOM': 2.3027, 'JUNGLE': 2.3255, 'MIDDLE': 2.1982, 'TOP': 2.4004, 'UTILITY': 3.1215}


In [5]:
# Distancia media ENTRE campeones de la misma línea (pares, no al centroide).
# Ojo: NO es lo mismo que la distancia al vector: con cuadrados vale la
# identidad E[||X-Y||^2] = 2·E[||X-mu||^2], pero con norma simple (sin
# cuadrar) la media entre pares es mayor que la media al centroide (Jensen).
pair_rows = []
for lane in dist_df["lane"].unique():
    vecs = np.stack([champLaneVectors[(c, lane)]
                     for c in dist_df[dist_df["lane"] == lane]["champion"]])
    Dp = pairwise_distances(vecs)
    iu = np.triu_indices(len(vecs), k=1)
    pair_rows.append({"lane": lane,
                      "n_mains": len(vecs),
                      "mean_pairwise": round(float(Dp[iu].mean()), 4),
                      "mean_to_centroid": round(float(mean_dist_per_lane[lane]), 4)})

pair_df = pd.DataFrame(pair_rows).sort_values("mean_pairwise")
print(pair_df.to_string(index=False))

# NOTA: este notebook no persiste nada (solo análisis; ver player_signatures.py).
print("OK: sin escritura a disco.")

   lane  n_mains  mean_pairwise  mean_to_centroid
 BOTTOM       26         2.4189            2.3027
 MIDDLE       32         2.7919            2.1982
 JUNGLE       41         3.0483            2.3255
    TOP       40         3.3190            2.4004
UTILITY       34         3.5147            3.1215
OK: sin escritura a disco.
